In [1]:
!pip install facenet-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2

In [1]:
import cv2
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from google.colab.patches import cv2_imshow
from PIL import Image
from torchvision import transforms
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
import os

# Obtenir le chemin du répertoire courant
chemin_courant = os.getcwd()

# Lister tous les fichiers et dossiers dans le répertoire courant
contenu = os.listdir(chemin_courant + '/sample_data')

# Afficher le contenu
print(f"Contenu du répertoire courant ('{chemin_courant}'):")
for element in contenu:
    print(element)

Contenu du répertoire courant ('/content'):
anscombe.json
README.md
mc_tatat_video.mp4
mc_tatat_face.jpg
california_housing_train.csv
california_housing_test.csv
mnist_test.csv
mnist_train_small.csv


In [7]:
def extraire_frames_par_seconde(video_path):
    """
    Extrait une frame par seconde d'une vidéo et les enregistre dans un dossier.

    Args:
        video_path (str): Le chemin d'accès au fichier vidéo.

    Returns:
        str: Le chemin d'accès au dossier où les frames sont enregistrées, ou None en cas d'erreur.
    """
    # Vérifier si le fichier vidéo existe
    if not os.path.exists(video_path):
        print(f"Erreur : Le fichier vidéo n'existe pas à l'emplacement : {video_path}")
        return None

    # Créer un objet VideoCapture pour lire la vidéo
    video_capture = cv2.VideoCapture(video_path)
    if not video_capture.isOpened():
        print(f"Erreur : Impossible d'ouvrir la vidéo à l'emplacement : {video_path}")
        return None

    # Obtenir le nombre de frames par seconde de la vidéo
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        print(f"Erreur : Impossible d'obtenir le nombre de frames par seconde de la vidéo.  Valeur de FPS : {fps}")
        return None

    # Créer le dossier de sortie
    video_name = os.path.splitext(os.path.basename(video_path))[0]  # Obtient le nom du fichier sans l'extension
    output_folder = f"{video_name}_frames"
    os.makedirs(output_folder, exist_ok=True)  # exist_ok=True pour éviter une erreur si le dossier existe déjà

    frame_count = 0
    saved_frame_count = 0
    while True:
        ret, frame = video_capture.read()
        if not ret:
            break  # Fin de la vidéo

        # Enregistrer une frame par seconde
        if frame_count % int(fps) == 0:
            frame_filename = f"frame_{saved_frame_count:04d}.jpg"  # Formatage pour avoir des noms de fichiers avec 4 chiffres
            frame_path = os.path.join(output_folder, frame_filename)
            cv2.imwrite(frame_path, frame)  # Enregistre la frame en tant qu'image JPEG
            saved_frame_count += 1

        frame_count += 1

    # Libérer les ressources
    video_capture.release()
    print(f"{saved_frame_count} frames ont été extraites et enregistrées dans le dossier : {output_folder}")
    return output_folder

In [15]:
def detect_and_recognize_faces_from_frames(frames_folder, output_video_name, reference_image_path, threshold=0.5):
    """
    Détecte et reconnaît un visage cible dans des frames, affiche un score de similarité,
    et génère une vidéo où chaque frame dure 1 minute.

    Args:
        frames_folder (str): Dossier contenant les images (frames).
        output_video_name (str): Nom de la vidéo de sortie.
        reference_image_path (str): Image du visage de référence à reconnaître.
        threshold (float): Seuil de similarité pour décider si c’est la même personne.

    Returns:
        str: Chemin vers la vidéo générée.
    """
    if not os.path.isdir(frames_folder):
        print(f"Erreur : Le dossier {frames_folder} n'existe pas.")
        return None

    # Initialiser les modèles
    # keep_all=True is important here as it returns a batched tensor even for a single face
    mtcnn = MTCNN(keep_all=True, image_size=160)
    resnet = InceptionResnetV1(pretrained='vggface2').eval()

    # Lire l'image de référence
    ref_img = Image.open(reference_image_path).convert('RGB')
    ref_face = mtcnn(ref_img)
    if ref_face is None:
        print("Erreur : Aucun visage détecté dans l'image de référence.")
        return None
    # ref_face is already a batched tensor from mtcnn(keep_all=True), so no need to unsqueeze(0)
    ref_embedding = resnet(ref_face)  # shape: [Number_of_Faces, 512]

    # Assuming the reference image contains only one prominent face, use the embedding of the first detected face
    # If multiple faces are expected in the reference image, you might need logic to select the correct one
    if ref_embedding.shape[0] > 1:
        print(f"Warning: Multiple faces detected in the reference image. Using the first one.")
    ref_embedding_single = ref_embedding[0].unsqueeze(0) # Take the first face's embedding and add a batch dim for cosine_similarity

    # Récupérer les frames
    image_files = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg', '.png'))])
    if not image_files:
        print("Erreur : Aucun fichier image trouvé dans le dossier.")
        return None

    # Dimensions pour la vidéo
    first_frame = cv2.imread(os.path.join(frames_folder, image_files[0]))
    height, width, _ = first_frame.shape
    fps = 1 / 1.0  # 1 frame = 1 min
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_video_name, fourcc, fps, (width, height))

    for idx, filename in enumerate(image_files):
        frame_path = os.path.join(frames_folder, filename)
        frame_bgr = cv2.imread(frame_path)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        # Use keep_all=True for detecting multiple faces in video frames
        boxes, _ = mtcnn.detect(frame_rgb)
        faces_in_frame = mtcnn(frame_rgb)


        if faces_in_frame is not None and boxes is not None:
            # Process each detected face in the current frame
            for i in range(faces_in_frame.shape[0]):
                embedding = resnet(faces_in_frame[i].unsqueeze(0)) # Add batch dim for single face embedding
                similarity = cosine_similarity(ref_embedding_single.detach().numpy(), embedding.detach().numpy())[0][0]

                # Get the corresponding bounding box for the face
                x1, y1, x2, y2 = map(int, boxes[i])

                # Dessiner rectangle rouge
                if similarity < threshold:
                  pass
                else:
                  # Dessiner rectangle rouge
                  cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
                  label = f"{similarity:.2f}"
                  if similarity >= threshold:
                      label = f"Match: {similarity:.2f}"
                      # Draw a green rectangle for matches
                      cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                      cv2.putText(frame_bgr, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                  else:
                      cv2.putText(frame_bgr, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)


        video_writer.write(frame_bgr)
        print(f"Frame {idx + 1}/{len(image_files)} traitée.")

    video_writer.release()
    print(f"✅ Vidéo générée : {output_video_name}")
    return output_video_name


In [16]:
# Chemin de la vidéo
chemin_courant = os.getcwd()
video_path = chemin_courant + '/sample_data/zeh_video.mp4'  # Remplacez par le chemin de votre fichier vidéo
# Chemin de l'image de référence
image_path = chemin_courant + '/sample_data/zeh_image.jpg'
# Extraction des frames
output_folder = extraire_frames_par_seconde(video_path)

# Vidéo de sortie
output_video_name = output_folder + '_delimited.mp4'

# Détection des visages
detect_and_recognize_faces_from_frames(output_folder, output_video_name, image_path)

21 frames ont été extraites et enregistrées dans le dossier : zeh_video_frames
Frame 1/21 traitée.
Frame 2/21 traitée.
Frame 3/21 traitée.
Frame 4/21 traitée.
Frame 5/21 traitée.
Frame 6/21 traitée.
Frame 7/21 traitée.
Frame 8/21 traitée.
Frame 9/21 traitée.
Frame 10/21 traitée.
Frame 11/21 traitée.
Frame 12/21 traitée.
Frame 13/21 traitée.
Frame 14/21 traitée.
Frame 15/21 traitée.
Frame 16/21 traitée.
Frame 17/21 traitée.
Frame 18/21 traitée.
Frame 19/21 traitée.
Frame 20/21 traitée.
Frame 21/21 traitée.
✅ Vidéo générée : zeh_video_frames_delimited.mp4


'zeh_video_frames_delimited.mp4'